In [2]:
import anndata as ad
import h5py
from scipy import sparse
import pandas as pd
import scanpy as sc

In [3]:
p4_adata = ad.read_h5ad("/blue/square.t/peter.huynh/jupyter/combined_samples_scvelo/p4/data/raw/p4_adata.h5ad")

In [4]:
f = h5py.File('/blue/square.t/peter.huynh/cellranger/cellranger_outs/cellranger_plus4_v5e_062522_forced6k/outs/plus4_filtered_feature_bc_matrix.h5', 'r')

barcodes = [b.decode() for b in f['matrix']['barcodes'][:]]
genes = [g.decode() for g in f['matrix']['features']['name'][:]] 

egfp_idx = genes.index("egfp") #a single number

data = f['matrix']['data'][:]
indices = f['matrix']['indices'][:]
indptr = f['matrix']['indptr'][:]
shape = f['matrix']['shape'][:]  
X = sparse.csc_matrix((data, indices, indptr), shape=shape)

egfp_vector = X[egfp_idx, :].toarray().flatten()#(row,column)
formatted_barcodes = ["plus4_v5e_062522_forced6k_possorted_genome_bam_CRXMJ:" + bc.replace("-1", "x") for bc in barcodes] 
egfp_series = pd.Series(egfp_vector, index=formatted_barcodes)
egfp_vector_aligned = egfp_series.reindex(p4_adata.obs_names).fillna(0).astype(int).values
egfp_var_idx = p4_adata.var_names.get_loc("egfp")
p4_adata.X = p4_adata.X.tolil()
p4_adata.X[:, egfp_var_idx] = egfp_vector_aligned.reshape(-1, 1)
p4_adata.X = p4_adata.X.tocsr()

In [9]:
print((egfp_vector_aligned > 0).sum())

60


In [10]:
##save point
p4_adata_egfp=p4_adata.copy()

In [11]:
adatas = {
    'p4': p4_adata_egfp
}

for name, adata in adatas.items():
    adata.var_names_make_unique()
    sc.pp.calculate_qc_metrics(adata, inplace=True)

In [12]:
#save this to processed
p4_adata_egfp.write("/blue/square.t/peter.huynh/jupyter/combined_samples_scvelo/p4/data/processed/p4_adata_egfp.h5ad")